In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

# 加载模型配置
# 请事先在 .env 中配置 DASHSCOPE_API_KEY
env_path = "/t9k/mnt/lxq/LangChain/dive-into-langgraph-main/.env"
_ = load_dotenv(dotenv_path=env_path)

/t9k/mnt/.conda/envs/LangChain/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# 配置大模型服务
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.0,  # 增加温度参数，确保输出稳定
)

# 创建一个简单的Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

# 运行Agent - 使用正确的消息格式
# 注意：create_agent 的 invoke 方法需要标准的消息列表格式
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好")  # 使用 HumanMessage 类包装用户输入
    ]
})

# 输出回复内容
print("Agent回复：", response['messages'][-1].content)

Agent回复： 你好！有什么我可以帮你的吗？


In [ ]:
# 一个工具函数
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# 创建带工具调用的Agent
tool_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# 运行Agent
response = tool_agent.invoke(
    {"messages": [HumanMessage(content="what is the weather in sf")]}   
)

print(response['messages'][-1].content)   




The weather in San Francisco (SF) is always sunny! 🌞


In [4]:
from typing import Literal, Any
from pydantic import BaseModel
from langchain.tools import tool, ToolRuntime

class Context(BaseModel):
    authority: Literal["admin", "user"]

# 创建带权限控制的tool，依赖ToolRuntime的内容进行判断
@tool
def math_add(runtime: ToolRuntime[Context, Any], a: int, b: int) -> int:
    """Add two numbers together."""
    authority = runtime.context.authority
    # 只有admin用户可以访问加法工具
    if authority != "admin":
        raise PermissionError("User does not have permission to add numbers")
    return a + b

# 创建带工具调用的Agent
tool_agent = create_agent(
    model=llm,
    tools=[get_weather, math_add],
    system_prompt="You are a helpful assistant",
)

# 在运行Agent时注入context
response = tool_agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"),
)

In [5]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

请计算 8234783 + 94123832 = ?
================================== Ai Message ==================================
Tool Calls:
  math_add (call_390eff112e114607a0d3eefb)
 Call ID: call_390eff112e114607a0d3eefb
  Args:
    a: 8234783
    b: 94123832
================================= Tool Message =================================
Name: math_add

102358615
================================== Ai Message ==================================

8234783 + 94123832 = 102358615。


In [6]:
# 验证计算结果是否正确
8234783 + 94123832

102358615

In [7]:
from pydantic import BaseModel, Field

class CalcInfo(BaseModel):
    """Calculation information."""
    output: int = Field(description="The calculation result")

In [8]:
# 创建带结构化输出的Agent
structured_agent = create_agent(
    model=llm,
    tools=[get_weather, math_add],
    system_prompt="You are a helpful assistant",
    response_format=CalcInfo,
)

response = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"),
)

In [9]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

请计算 8234783 + 94123832 = ?
================================== Ai Message ==================================
Tool Calls:
  math_add (call_f7ee1bbaf0d040e9bdf78401)
 Call ID: call_f7ee1bbaf0d040e9bdf78401
  Args:
    a: 8234783
    b: 94123832
================================= Tool Message =================================
Name: math_add

102358615
================================== Ai Message ==================================
Tool Calls:
  CalcInfo (call_e4189f7157284b9ead223a2a)
 Call ID: call_e4189f7157284b9ead223a2a
  Args:
    output: 102358615
================================= Tool Message =================================
Name: CalcInfo

Returning structured response: output=102358615


In [10]:
response['messages'][-1]

ToolMessage(content='Returning structured response: output=102358615', name='CalcInfo', id='ab05fff2-88b1-4e49-8a3e-f500a56cb679', tool_call_id='call_e4189f7157284b9ead223a2a')

In [11]:
agent = create_agent(
    model=llm,
    tools=[get_weather],
)

for chunk in agent.stream(  
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'call_c1e84ffc72bc4be5a99c5ecd'}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
step: model
content: [{'type': 'text', 'text': "It seems there might be some confusion. While San Francisco (SF) is known for its microclimates and can have varying weather, it's not always sunny. The weather can range from foggy and cool to partly cloudy or sunny, especially during different times of the year. If you'd like, I can check the current weather conditions for you!"}]
